<a href="https://colab.research.google.com/github/HasanKhatib/iot-playground/blob/main/autdio_processing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Abstract

Hey there! This project is all about setting up a straightforward audio classification pipeline in Google Colab. By loading `.wav` files and parsing `info.labels` from Edge Impulse, we build a custom PyTorch dataset, apply a CNN model (M5), and train it to recognize words like “apple,” “orange,” “cherry,” and “unknown.” Everything’s done step-by-step (from mounting Drive to running the final evaluation) so it’s easy to follow and get solid accuracy on your test data.

Check the sections below 👇 for more detailed guide and then for the solution.


# Readme.md

## Friendly Engineer's Guide to Audio Classification

Hey there! This is my personal engineering log (and pep talk) on how I set up the audio classification pipeline in Google Colab. Below is a **step-by-step** breakdown of the entire process, from mounting my Google Drive all the way to training and evaluating my model.

### Step 1: Mount Google Drive
1. Open a Colab notebook.
2. Use `drive.mount('/content/drive')` to hook up my Drive so I can access all the files in `/content/drive/MyDrive`.
3. Verify that the `IoT/training` and `IoT/testing` folders are in the right place.

> **Why?** Because I need to read the `.wav` files and `info.labels` file directly from Drive.

---

### Step 2: Parse `info.labels`
1. I load the JSON file using Python’s built-in `json` module.
2. It creates a dictionary mapping each `.wav` filename to a text label (like "apple," "orange," etc.).
3. I check how many items are there, just to confirm I’m reading the file properly.

> **Why?** Because my dataset comes from Edge Impulse, and that `info.labels` tells me which audio belongs to which class.

---

### Step 3: Create a PyTorch Dataset
1. Define a custom `EdgeImpulseAudioDataset` that takes:
   - A folder path (like `/content/drive/MyDrive/IoT/training`)
   - The label dictionary from Step 2
   - A label map (e.g., `"apple":0, "orange":1, ...`)
2. In `__getitem__`, I load the `.wav` file with `torchaudio.load` and fetch its label.

> **Why?** Because I need a standard interface for PyTorch to grab `(waveform, label)` pairs.

---

### Step 4: Build DataLoaders
1. I instantiate `train_dataset` and `test_dataset` with the custom dataset class.
2. Then I wrap them with `DataLoader` (e.g., `train_loader`, `test_loader`).
3. **Important**: I add a `collate_fn` to handle variable waveform lengths by padding them (e.g., `pad_collate`).

> **Why?** Because PyTorch expects to stack each batch of waveforms into a single tensor. If they’re different lengths, it can’t do that unless I pad or truncate.

---

### Step 5: Define My CNN Model (M5)
1. I set up the **M5** architecture:
   - Convolution layers with kernel sizes, strides.
   - Batch normalization and pooling.
   - A final fully connected layer to map to 4 classes (`apple`, `orange`, `cherry`, `unknown`).
2. I place the model on `cuda` if available, so I can use the GPU.

> **Why?** Because M5 is a straightforward 1D CNN suitable for raw audio data. It’s simpler than other advanced architectures but good enough for this lab.

---

### Step 6: Loss and Optimizer
1. I pick `nn.CrossEntropyLoss()` since it’s classification.
2. I choose `optim.Adam(model.parameters())` with `lr=0.001`.
3. This is standard practice for many classification tasks in PyTorch.

> **Why?** Cross-entropy is the go-to for multi-class classification, and Adam is typically a solid default optimizer.

---

### Step 7: Train the Model
1. Loop over epochs (like 50 or more).
2. For each batch (`waveforms`, `labels`) from `train_loader`:
   - **Zero** out the gradients.
   - **Forward** pass through the model.
   - **Compute** the loss.
   - **Backward** pass to compute gradients.
   - **Step** the optimizer to update weights.
3. Keep track of the loss, watch it go down over time.

> **Why?** Because that’s standard procedure for training neural networks: forward, backward, optimize, repeat.

---

### Step 8: Evaluation (The Next Big Moment!)
1. Now that everything’s trained, I switch to `model.eval()` to disable dropout and batchnorm updates.
2. I loop over `test_loader` with `torch.no_grad()`, so I don’t track gradients during evaluation.
3. For each test batch, I:
   - Predict the class with `model(waveforms)`.
   - Compare with the true labels.
   - Keep a running count of how many predictions are correct vs. the total.
4. Finally, I print the accuracy (correct / total * 100).

> **Why?** Because I need to see how well my model generalizes to unseen data and confirm that it correctly classifies “apple,” “orange,” “cherry,” and “unknown.”

---

### Closing Thoughts
- If I encounter shape mismatch errors, I **pad** or **truncate** waveforms in the `collate_fn`.
- If I get a `KeyError` for a label, I either add that label to `label_map` or rename it in the dataset.
- Once I see a nice accuracy at Step 8, I know my pipeline works!

**That’s all, folks!** This text box helps me remember each step and keep track of what’s going on. If I get stuck, I come back here, see where I left off, and tackle the issue systematically. Now I can proceed confidently to run evaluations and fine-tune if needed.


# Solution

In [5]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [30]:
import json
import os

labels_train = {}
labels_test = {}

# Paths to info.labels in training & testing
info_train_path = '/content/drive/MyDrive/iot/training/info.labels'
info_test_path = '/content/drive/MyDrive/iot/testing/info.labels'

def read_labels(json_path):
    with open(json_path, 'r') as f:
        data = json.load(f)
    # data['files'] is a list of dicts
    label_dict = {}
    for item in data['files']:
        # e.g. "apple.5l3vikl6.ingestion-...wav"
        filename = item['path']
        label = item['label']['label']  # e.g. "apple"
        label_dict[filename] = label
    return label_dict

labels_train = read_labels(info_train_path)
labels_test = read_labels(info_test_path)

# Check how many labeled files
print("Training samples:", len(labels_train))
print("Testing samples:", len(labels_test))


Training samples: 67
Testing samples: 17


In [31]:
import torch.nn.functional as F

def pad_collate(batch):
    """
    batch: list of (waveform, label)
    waveform shape: [1, waveform_length]
    label: int
    """
    # 1) Find the longest waveform in this batch
    max_length = max(waveform.shape[-1] for waveform, _ in batch)

    waveforms_padded = []
    labels = []

    for waveform, label in batch:
        length = waveform.shape[-1]
        if length < max_length:
            # Zero-pad (right side)
            pad_amount = max_length - length
            waveform = F.pad(waveform, (0, pad_amount))
        # If you also want to truncate waveforms that are too long, you could do that here
        waveforms_padded.append(waveform)
        labels.append(label)

    # Stack into a batch
    waveforms_padded = torch.stack(waveforms_padded, dim=0)  # [batch_size, 1, max_length]
    labels = torch.tensor(labels, dtype=torch.long)
    return waveforms_padded, labels


In [32]:
import torch
import torchaudio
from torch.utils.data import Dataset, DataLoader

# Map text labels to numeric IDs
label_map = {
    "apple": 0,
    "orange": 1,
    "cherry": 2,
    "unknown": 3
}

class EdgeImpulseAudioDataset(Dataset):
    def __init__(self, data_dir, label_dict, label_map):
        """
        data_dir: path to folder with wav files (e.g. /content/drive/MyDrive/iot/training)
        label_dict: dict of { filename_in_info.labels: label_str }
        label_map: dict mapping label_str -> integer class
        """
        self.data_dir = data_dir
        self.label_dict = label_dict
        self.label_map = label_map

        # Collect all .wav files that appear in label_dict
        self.wav_files = []
        for f in os.listdir(self.data_dir):
            if f.endswith('.wav') and f in self.label_dict:
                self.wav_files.append(f)

    def __len__(self):
        return len(self.wav_files)

    def __getitem__(self, idx):
        wav_name = self.wav_files[idx]
        wav_path = os.path.join(self.data_dir, wav_name)

        # Load audio
        waveform, sample_rate = torchaudio.load(wav_path)

        # Convert to mono if multiple channels
        # shape: [1, num_samples]
        if waveform.shape[0] > 1:
            waveform = torch.mean(waveform, dim=0, keepdim=True)

        # Get the string label from label_dict, then map to int
        label_str = self.label_dict[wav_name]
        label = self.label_map[label_str]

        return waveform, label


In [33]:
from torch.utils.data import DataLoader

# Paths to your audio folders
train_dir = '/content/drive/MyDrive/iot/training'
test_dir  = '/content/drive/MyDrive/iot/testing'

train_dataset = EdgeImpulseAudioDataset(train_dir, labels_train, label_map)
test_dataset = EdgeImpulseAudioDataset(test_dir, labels_test, label_map)

# Create DataLoaders (adjust batch_size as needed)
train_loader = DataLoader(
    train_dataset,
    batch_size=8,
    shuffle=True,
    collate_fn=pad_collate
)

test_loader = DataLoader(
    test_dataset,
    batch_size=8,
    shuffle=False,
    collate_fn=pad_collate
)


print("Train dataset size:", len(train_dataset))
print("Test dataset size:", len(test_dataset))


Train dataset size: 67
Test dataset size: 17


In [34]:
import torch.nn as nn
import torch.nn.functional as F

class M5(nn.Module):
    def __init__(self, n_input=1, n_output=4, stride=16, n_channel=32):
        super().__init__()
        self.conv1 = nn.Conv1d(n_input, n_channel, kernel_size=80, stride=stride)
        self.bn1 = nn.BatchNorm1d(n_channel)
        self.pool1 = nn.MaxPool1d(4)

        self.conv2 = nn.Conv1d(n_channel, n_channel, kernel_size=3)
        self.bn2 = nn.BatchNorm1d(n_channel)
        self.pool2 = nn.MaxPool1d(4)

        self.fc1 = nn.Linear(n_channel, n_output)

    def forward(self, x):
        # x shape: [batch_size, 1, num_samples]
        x = self.conv1(x)
        x = self.bn1(x)
        x = F.relu(x)
        x = self.pool1(x)

        x = self.conv2(x)
        x = self.bn2(x)
        x = F.relu(x)
        x = self.pool2(x)

        # global average pooling (mean over time dim)
        x = torch.mean(x, dim=2)

        x = self.fc1(x)
        return x


In [35]:
import torch.optim as optim

model = M5(n_input=1, n_output=4)  # 1 input channel, 4 classes
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)


In [36]:
num_epochs = 2  # Increase to 50 or more in practice

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0

    for waveforms, labels in train_loader:
        waveforms = waveforms.to(device)          # shape: [batch_size, 1, num_samples]
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(waveforms)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    epoch_loss = running_loss / len(train_loader)
    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {epoch_loss:.4f}")


Epoch [1/2], Loss: 1.3529
Epoch [2/2], Loss: 1.2645


In [37]:
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for waveforms, labels in test_loader:
        waveforms = waveforms.to(device)
        labels = labels.to(device)

        outputs = model(waveforms)
        _, predicted = torch.max(outputs, dim=1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = 100.0 * correct / total
print(f"Test Accuracy: {accuracy:.2f}%")


Test Accuracy: 23.53%
